# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## A logistic regression model, built from parts (after class)

In class you built a `Linear` layer, one method at a time: a **forward** pass, a **backward** pass and an **update**. In this workbook you finish the classifier. You write the **loss** (two functions, `bce_loss` and `bce_grad`), and then put everything together in a **training loop** like this one, in which each line does one job:

```python
for epoch in range(n_epochs):
    scores = model.forward(X)                 # the forward pass
    loss = bce_loss(scores, y)                # how wrong is it?
    model.backward(bce_grad(scores, y))       # the backward pass: the gradients
    model.update(lr)                          # change the weights
```

Using and evaluating the model you train here, and comparing several models, comes at the start of Week 9.

| Part | Topic |
|---|---|
| 0 | Set-up: the data and the layer from class |
| 1 | The loss |
| 2 | Training: one step, then many |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Set-up

The next two cells set up everything from the in-class workbook, so that this workbook works on its own: the simulated data `X` and `y` (50 texts, each with two features, the log counts of positive and of negative words), the helper functions `dot`, `mat_vec` and `transpose`, and the `Linear` class. Read through them to remind yourself of each piece, then run them.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
X = [[x[0][k], x[1][k]] for k in range(len(y))]

In [ ]:
def dot(a, b):
    # the dot product of two lists of the same length
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X                       # kept for the backward pass
        scores = []
        for x in X:
            z = mat_vec(self.weight, x)
            for c in range(len(z)):
                z[c] += self.bias[c]
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):
            for c in range(n_out):
                for j in range(n_in):
                    self.weight_grad[c][j] += grad[k][c] * self.inputs[k][j]
                self.bias_grad[c] += grad[k][c]
        # the gradients for the inputs, for a layer that comes before this one
        return [mat_vec(transpose(self.weight), g) for g in grad]

    def update(self, lr):
        for c in range(len(self.weight)):
            for j in range(len(self.weight[c])):
                self.weight[c][j] -= lr * self.weight_grad[c][j]
            self.bias[c] -= lr * self.bias_grad[c]

---
# Part 1 · The loss

The loss measures how wrong the model's scores are. For two classes we use the cross-entropy loss from LELA60331 Week 5 (sometimes called **binary cross-entropy**, or BCE). A layer with one output gives one row per example with one score in it, so the score of example `k` is `scores[k][0]`.

For each example, the loss turns the score into a probability with `sigmoid`, then takes minus the log of the probability given to the correct label. The loss for all the examples is the average. We build it in three small steps: the probability, the loss for one example, and then the average for all.

### The tool: `sigmoid`

The `sigmoid` function from LELA60331 Week 5 turns any score into a probability between 0 and 1.

🔮 **Predict**: what is `sigmoid(0)`? Will `sigmoid(2)` be above or below it? Run the cell to define it and check.

In [ ]:
# ▶ Run this cell: the helper function for the loss
def sigmoid(z):
    return 1 / (1 + math.exp(-z))

print(sigmoid(0), round(sigmoid(2), 3), round(sigmoid(-2), 3))

### 🏋️ Exercise 1 · Practice — The loss for one example

Take one example with a score of 2.0.

1. `q_example`: its probability of being positive
2. `loss_if_positive`: the loss if its label is 1, that is −log₂ of the probability given to the label 1
3. `loss_if_negative`: the loss if its label is 0. *Hint:* the probability given to the label 0 is `1 - q_example`.

🔮 **Predict**: which of the two losses will be bigger? Why?

In [ ]:
q_example = ???                                                # 1.
loss_if_positive = ???                                         # 2.
loss_if_negative = ???                                         # 3.
print(round(q_example, 3), round(loss_if_positive, 3), round(loss_if_negative, 3))   # you should get 0.881 0.183 3.069

In [ ]:
# ✅ Check
check("1. q_example", lambda: round(q_example, 3) == 0.881)
check("2. loss_if_positive", lambda: round(loss_if_positive, 3) == 0.183)
check("3. loss_if_negative", lambda: round(loss_if_negative, 3) == 3.069)

### 🏋️ Exercise 2 · Practice — The loss for all the examples

Now put the same steps in a function, `bce_loss(scores, y)`, which returns the average loss over all the examples. The `if` and `else` (your steps 2 and 3, with a tiny number `eps` added so that we never take the log of 0) are done for you.

1. `q`: the probability for example `k`. *Hint:* its score is `scores[k][0]`.
2. the average of `losses`

In [ ]:
def bce_loss(scores, y):
    eps = 0.00001          # a tiny number, so that we never take the log of 0
    losses = []
    for k in range(len(y)):
        q = ???                                               # 1. the probability
        if y[k] == 1:
            losses.append(-math.log2(q + eps))
        else:
            losses.append(-math.log2(1 - q + eps))
    return ???                                                # 2. the average

print(round(bce_loss([[2.0]], [1]), 3))                        # one example, label 1: the same as loss_if_positive, 0.183
print(round(bce_loss([[0.0], [2.0]], [1, 0]), 3))              # two examples: you should get 2.034, the average of 1.0 and 3.069

In [ ]:
# ✅ Check
check("bce_loss, one example", lambda: round(bce_loss([[2.0]], [1]), 3) == 0.183)
check("bce_loss, two examples", lambda: round(bce_loss([[0.0], [2.0]], [1, 0]), 3) == 2.034)

### The gradient of the loss

The backward pass needs one more thing from the loss: for each example, how much the loss changes when that example's score goes up. For this loss, that is exactly the **error** from LELA60331 Week 5, divided by the number of examples (because the loss is an average):

`(q - y[k]) / n`, where `q` is the sigmoid of example `k`'s score

For the two examples above (scores 0.0 and 2.0, labels 1 and 0, so `n` = 2):

| example | `q` | label | `(q - label) / 2` |
|---|---|---|---|
| 0 | 0.5 | 1 | −0.25 |
| 1 | 0.881 | 0 | 0.44 |

A negative gradient means the score should go **up** (towards the label 1); a positive one means it should go **down**.

`bce_grad` returns these as a matrix with one row per example and one number in each row, the same shape as the scores, so that it can be passed straight to the layer's `backward`.

### 🏋️ Exercise 3 · Practice — `bce_grad`

1. `grad_example`: first, by hand, the gradient for example 1 in the table (score 2.0, label 0, `n` = 2)
2. complete the loop in `bce_grad(scores, y)`: for each example, add a row with one number, `(q - y[k]) / n`. *Hint:* a row with one number in it is written `[...]`.

In [ ]:
grad_example = ???                                            # 1.
print(round(grad_example, 3))                                  # you should get 0.44

def bce_grad(scores, y):
    n = len(y)
    grad = []
    for k in range(n):
        q = sigmoid(scores[k][0])
        grad.append(???)                        # 2. a row with one number
    return grad

print([[round(v, 3) for v in row] for row in bce_grad([[0.0], [2.0]], [1, 0])])   # you should get [[-0.25], [0.44]]

In [ ]:
# ✅ Check
check("1. grad_example", lambda: round(grad_example, 3) == 0.44)
check("2. bce_grad", lambda: [[round(v, 3) for v in row] for row in bce_grad([[0.0], [2.0]], [1, 0])] == [[-0.25], [0.44]])
check("2. one row per example, with one number", lambda: [len(row) for row in bce_grad([[0.0], [2.0], [1.0]], [1, 0, 1])] == [1, 1, 1])

---
# Part 2 · Training: one step, then many

Logistic regression is just a `Linear` layer with one output, trained with this loss. As in LELA60331 Week 5, we first make **one** improvement, step by step, and check that the loss goes down. Then we repeat it a few times, and finally many times.

### 🏋️ Exercise 4 · Practice — One step, part 1: how wrong is the model?

1. `model`: a `Linear` layer with 2 inputs and 1 output (after `random.seed(10)`, so that everyone gets the same starting weights as in LELA60331 Week 7)
2. `scores`: the scores of every text in `X` (the forward pass)
3. `loss_before`: the loss

In [ ]:
random.seed(10)
model = ???                                             # 1.
print("weights:", model.weight, "bias:", model.bias)
scores = ???                                            # 2.
print("the first three scores:", scores[:3])
loss_before = ???                                       # 3.
print("loss:", round(loss_before, 3))                   # you should get 2.455

In [ ]:
# ✅ Check
check("1. one output, two inputs", lambda: len(model.weight) == 1 and len(model.weight[0]) == 2)
check("2. one score per text", lambda: len(scores) == 50 and len(scores[0]) == 1)
check("3. loss_before", lambda: abs(loss_before - 2.455) < 0.001)

### 🏋️ Exercise 5 · Practice — One step, part 2: improve it

4. the backward pass: give `model.backward` the gradients from `bce_grad`
5. the update, with a learning rate of 0.5

🔮 **Predict**: the cell prints the gradients and the new weights. The first scores in Exercise 4 were all well above 0 (a probability near 1), but fewer than half the texts are positive. Will the gradient for the bias be positive or negative, and so will the bias go up or down?

In [ ]:
???                                                     # 4.
print("gradients:", model.weight_grad, model.bias_grad)
???                                                     # 5.
print("new weights:", model.weight, "new bias:", model.bias)

### 🏋️ Exercise 6 · Practice — One step, part 3: is it better?

6. `loss_after`: the loss of the updated model. *Hint:* you need new scores, from a new forward pass.

In [ ]:
loss_after = ???                                        # 6.
print(round(loss_before, 3), round(loss_after, 3))      # you should get 2.455 0.866

In [ ]:
# ✅ Check
check("4. backward has been run", lambda: hasattr(model, "weight_grad"))
check("6. the loss went down", lambda: loss_after < loss_before)
check("6. loss_after", lambda: abs(loss_after - 0.866) < 0.001)

### 🏋️ Exercise 7 · Practice — A few steps in a loop

Now put the step in a loop. Each pass through the data (each **epoch**) does the same four things as Exercises 4–6:

1. calculates the scores (the **forward pass**)
2. calculates the loss, and records it
3. calculates the gradients (the **backward pass**)
4. updates the weights and the bias

Start with a new model and just **5** epochs, printing the loss each time, so that you can see each step.

In [ ]:
random.seed(10)
model = Linear(2, 1)
lr = 0.5

for epoch in range(5):
    scores = ???                                 # 1.
    loss = ???                                   # 2.
    print("epoch", epoch, "loss", round(loss, 3))
    ???                                          # 3.
    ???                                          # 4.
# you should get 2.455, then 0.866, then losses that keep going down

In [ ]:
# ✅ Check
check("the loss in the last epoch is lower than the first", lambda: loss < loss_before)

### 🏋️ Exercise 8 · Practice — The full training loop

Now train properly, with the settings of LELA60331 Week 7: a learning rate of 0.5 and 1000 epochs, with a new model made after `random.seed(10)`. Fill in the same four lines as in Exercise 7. This time the losses are kept in `loss_history` and plotted, rather than printed.

In [ ]:
random.seed(10)
model = Linear(2, 1)
lr = 0.5
n_epochs = 1000
loss_history = []

for epoch in range(n_epochs):
    scores = ???                                 # 1.
    loss = ???                                   # 2.
    loss_history.append(loss)
    ???                                          # 3.
    ???                                          # 4.

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print(model.weight, model.bias)                # about [[1.17, -1.32]] and [0.51]
print("final loss:", round(loss_history[-1], 3))    # you should get 0.656

In [ ]:
# ✅ Check
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("the first loss is the one from Exercise 4", lambda: abs(loss_history[0] - loss_before) < 1e-12)
check("the same weights as LELA60331 Week 7", lambda: [round(w, 2) for w in model.weight[0]] == [1.17, -1.32] and round(model.bias[0], 2) == 0.51)
check("final loss about 0.656", lambda: abs(loss_history[-1] - 0.656) < 0.001)

These are the weights from LELA60331 Week 7. Compare the two pieces of code: which lines in the seminar's training loop does each line of this loop replace?

---
## ✏️ Summary

You can now:

* explain why a program with many shared global names is fragile, and how a class helps
* build a class one method at a time, testing each method on a small example
* write a `Linear` layer with a **forward** pass, a **backward** pass and an **update**
* write a loss as two functions: one for its value, and one for its gradient
* make one training step by hand, then train a model with a loop of four lines: forward, loss, backward, update

**After class:** work through `LELA60341_Wk8_after_class`, which uses the model you have built here.

---
## ✏️ Summary

You can now:

* write a loss as two functions: one for its value, and one for its gradient
* make one training step by hand: forward, loss, backward, update, and check that the loss went down
* train a model with a loop of four lines, one for each of those steps

**Next week:** Week 9 starts by using the model you have trained here: its probabilities and predictions, evaluating it, and comparing several models.